# 04. PyRPMD GPU 加速与 PyTorch 张量动力学模块案例 (GPU Acceleration & Tensor Backend)

本教程演示 **PyRPMD** 的设备驻留（Device-Resident）PyTorch 张量动力学与批量加速模块：

1. **自动设备选择与设备驻留张量动力学 (`TorchPES`, `TorchState`, `TorchDynamics`)**：
   - 自动检测 `cuda`（若当前机器存在 NVIDIA GPU）或使用 `cpu` 进行双精度（`torch.float64`）验证；
   - 状态张量 `q, p` 形状为 `[B, P, N, 3]`（$B$ 条独立轨迹、$P$ 个珠子、$N$ 个原子、$3$ 个笛卡尔分量），整个 `propagate` 块内正规模变换（`rfft` / `rpmdrate`）、Cayley 自由环演化、BCOCB/PILE-L 热浴更新与势能求导全部留在设备上执行，无逐步 `.cpu()` 或 `.item()` 同步开销。
2. **可微周期性 PyTorch 势能面与自动微分应力维里 (`PeriodicCosine` `torch.nn.Module`)**：
   - 定义支持分数坐标、周期性边界条件（PBC）与晶胞体积项的 PyTorch 模型，通过坐标反向传播自动计算解析力，并通过晶胞应变导数计算维里张量（Virial）。
3. **张量材料工作流（`sampling.engine = "torch"`）与原子级断点续算 (`resume_run`)**：
   - 在 `pimd` 与 `diffusion` 任务中启用 `sampling.engine = "torch"`，验证与有限珠数谐振子解析量子统计精确解的一致性，并演示单文件原子级批次检查点（`checkpoints/materials_torch.npz`）的无损续算。
4. **跨轨迹批量合并评估与吞吐量对比 (`TorchDynamics` Batch Scaling)**：
   - 将 $B$ 条独立环聚合物轨迹的 $B \times P$ 个珠子构型合并为单次 `[B*P, N, 3]` 张量势能调用，对比不同批量大小 $B$ 下的单步吞吐量提升。

In [ ]:
from pathlib import Path
import tempfile
import time
import json
import numpy as np
import torch

import pyrpmd
from pyrpmd import JobSpec, System
from pyrpmd.runner import run, resume_run
from pyrpmd.torch_backend import TorchPES, TorchState, TorchDynamics
from pyrpmd.batched import BatchedDynamics, BatchedState
from pyrpmd.potentials import HarmonicPotential

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path(pyrpmd.__file__).resolve().parents[2]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyRPMD version: {pyrpmd.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Selected execution device: {DEVICE} (CUDA available: {torch.cuda.is_available()})")

## 1. 设备驻留张量动力学：`TorchPES`、`TorchState` 与 `cayley-bcocb` 采样

使用 `integrator="cayley-bcocb", thermostat="pile-l"` 时，单步传播算符采用 Korol et al. (*J. Chem. Phys.* **152**, 104102, 2020) 的主平方根分裂：
$$\exp(\Delta t\,\mathcal{L}) \approx B(\Delta t/2)\,\sqrt{\operatorname{cay}(\Delta t A)}\,O(\Delta t)\,\sqrt{\operatorname{cay}(\Delta t A)}\,B(\Delta t/2)$$
在物理力稳定步长 $\Delta t \sqrt{k/m} < 2$ 内，该分裂对任意珠子数 $P$ 均保持**精确的有限珠数谐振子位置边缘分布**：
$$\langle q^2 \rangle_P = \frac{1}{\beta m}\sum_{k=0}^{P-1} \frac{1}{\omega^2 + \omega_k^2}, \quad \omega_k = \frac{2P}{\beta}\sin\left(\frac{\pi k}{P}\right)$$

In [ ]:
class TensorHarmonicModel(torch.nn.Module):
    metadata = {"batch_independent": True}

    def __init__(self, k=2.0):
        super().__init__()
        self.k = float(k)

    def forward(self, q):
        # 输入 q 形状: [frames, natoms, 3]，返回每个 frame 的总势能 [frames]
        return 0.5 * self.k * q.square().sum(dim=(1, 2))


k_spring = 2.0
mass_au = 2.0
beta_au = 4.0
nbeads = 16
n_trajectories = 16

pes = TorchPES(TensorHarmonicModel(k=k_spring), device=DEVICE)
dynamics = TorchDynamics(
    pes,
    masses=[mass_au],
    beta=beta_au,
    nbeads=nbeads,
    dt=0.25,
    integrator="cayley-bcocb",
    thermostat="pile-l",
    gamma_centroid=1.0,
    seed=2026,
)

# 初始化 [B=16, P=16, N=1, 3] 张量状态并在设备上块式传播
q0 = torch.zeros((n_trajectories, nbeads, 1, 3), dtype=torch.float64, device=pes.device)
state = TorchState(q0, dynamics.sample_momenta(n_trajectories))

# 平衡 200 步后，采样 20 个块（每块 25 步）统计位置方差
dynamics.propagate(state, 200)
q2_samples = []
for _ in range(20):
    dynamics.propagate(state, 25)
    q2_samples.append(state.q.square().mean().item())

measured_q2 = float(np.mean(q2_samples))
omega = np.sqrt(k_spring / mass_au)
omega_k = 2.0 * nbeads / beta_au * np.sin(np.pi * np.arange(nbeads) / nbeads)
exact_q2 = float(np.sum(1.0 / (omega**2 + omega_k**2)) / (beta_au * mass_au))

print(json.dumps({
    "device": str(pes.device),
    "state_shape": list(state.q.shape),
    "measured_mean_q2_bohr2": measured_q2,
    "exact_finite_bead_q2_bohr2": exact_q2,
    "relative_error": abs(measured_q2 - exact_q2) / exact_q2,
}, indent=2))

assert abs(measured_q2 - exact_q2) / exact_q2 < 0.08

## 2. 可微周期性 PyTorch 势能面与自动微分维里 (`periodic_torch.inp`)

在固体或凝聚相模拟中，PyTorch 模型不仅可以对原子坐标求导得到力 $\mathbf{F} = -\partial V / \partial \mathbf{q}$，还可以直接接收晶胞矩阵 `cell` 与周期性标志 `pbc`，通过自动微分计算应变维里张量（Strain Virial）。下面我们运行 `examples/potentials/periodic_torch.inp` 验证周期性 PyTorch 势能工作流。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    pt_spec = JobSpec.from_file(ROOT / "examples/potentials/periodic_torch.inp")
    pt_spec.data["potential"]["device"] = DEVICE
    pt_spec.data["sampling"].update(
        equilibration="2 fs", production="6 fs", interval="0.4 fs", replicas=2
    )
    run(pt_spec, output=tmp_path / "periodic_torch")
    pt_summary = json.loads((tmp_path / "periodic_torch/results/summary.json").read_text())

print(json.dumps({
    "task": pt_summary["task"],
    "status": pt_summary["status"],
    "pbc": pt_summary["pbc"],
    "estimators": pt_summary["estimators"],
}, indent=2))
assert pt_summary["status"] == "complete" 

## 3. 张量材料工作流（`sampling.engine = "torch"`）与原子级断点续算

在 `pimd` 和 `diffusion` 工作流中设置 `sampling.engine = "torch"` 时，PyRPMD 会将全部 `replicas = B` 条轨迹打包为单个设备驻留张量 `[B, P, N, 3]`，并使用 `checkpoints/materials_torch.npz` 进行原子级提交。当已完成的运行调用 `resume_run` 时，无需重新构造势能面或消耗随机数即可无损重放导出结果。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    torch_pimd = JobSpec.from_file(ROOT / "examples/materials/torch_pimd.inp")
    torch_pimd.data["sampling"].update(
        device=DEVICE,
        equilibration_steps=32,
        production_steps=128,
        sample_interval=8,
        replicas=8,
    )
    torch_pimd.data["output"].update(checkpoint_steps=64)

    out_dir = tmp_path / "torch_pimd_run"
    run(torch_pimd, output=out_dir)

    # 验证原子级检查点文件存在并执行断点重放/续算
    ckpt_file = out_dir / "checkpoints/materials_torch.npz"
    checkpoint_exists = ckpt_file.is_file()
    assert checkpoint_exists
    resume_run(out_dir)

    tp_summary = json.loads((out_dir / "results/summary.json").read_text())

print(json.dumps({
    "checkpoint_exists": checkpoint_exists,
    "replicas": tp_summary["replicas"],
    "samples_per_replica": tp_summary["samples_per_replica"],
    "potential_energy_mean": tp_summary["estimators"]["potential_energy"]["mean"],
    "kinetic_cv_mean": tp_summary["estimators"]["kinetic_centroid_virial"]["mean"],
    "radius_of_gyration_squared_bohr2": float(np.mean(tp_summary["radius_of_gyration_squared_bohr2"])),
}, indent=2))

## 4. 跨轨迹张量批量化吞吐量对比 (`TorchDynamics` Batch Scaling)

在 GPU 或多核矢量硬件上，逐条轨迹串行调用势能面会受限于内核启动与 Python 调度开销；将 $B$ 条独立环聚合物轨迹合并为形状 `[B, P, N, 3]` 的单次张量块传播，可在每一步用单次 `[B*P, N, 3]` 评估完成全部轨迹与珠子的力计算。下面我们测量不同轨迹批量大小 $B \in \{1, 8, 32\}$ 下每步每条轨迹的平均耗时与吞吐量（trajectories $\times$ steps / s）。

In [ ]:
beads_bench = 16
steps_bench = 80
bench_pes = TorchPES(TensorHarmonicModel(k=1.0), device=DEVICE)

# 预热一次消除首次 FFT 与算子初始化开销
warm_dyn = TorchDynamics(
    bench_pes, masses=[1836.15, 1836.15, 1836.15], beta=8.0, nbeads=beads_bench,
    dt=0.2, integrator="cayley-bcocb", thermostat="pile-l", gamma_centroid=0.5, seed=1
)
warm_state = TorchState(
    torch.zeros((2, beads_bench, 3, 3), dtype=torch.float64, device=bench_pes.device),
    warm_dyn.sample_momenta(2),
)
warm_dyn.propagate(warm_state, 10)

scaling_records = []
for batch_size in (1, 8, 32):
    dyn_b = TorchDynamics(
        bench_pes,
        masses=[1836.15, 1836.15, 1836.15],
        beta=8.0,
        nbeads=beads_bench,
        dt=0.2,
        integrator="cayley-bcocb",
        thermostat="pile-l",
        gamma_centroid=0.5,
        seed=100 + batch_size,
    )
    st_b = TorchState(
        torch.zeros((batch_size, beads_bench, 3, 3), dtype=torch.float64, device=bench_pes.device),
        dyn_b.sample_momenta(batch_size),
    )
    if DEVICE.startswith("cuda"):
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    dyn_b.propagate(st_b, steps_bench)
    if DEVICE.startswith("cuda"):
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    traj_steps_per_sec = (batch_size * steps_bench) / elapsed
    scaling_records.append({
        "batch_trajectories": batch_size,
        "total_bead_frames_per_step": batch_size * beads_bench,
        "wall_seconds": round(elapsed, 4),
        "trajectory_steps_per_second": round(traj_steps_per_sec, 1),
    })

print(json.dumps({
    "device": DEVICE,
    "beads": beads_bench,
    "steps": steps_bench,
    "batch_scaling": scaling_records,
}, indent=2))
assert scaling_records[-1]["trajectory_steps_per_second"] > scaling_records[0]["trajectory_steps_per_second"]